# Example-02: Singular values

In [1]:
# Import

import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import data_load
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# svd_list and svd_list_randomized can be used to compute singular values
# svd_list performs full SVD and svd_list_randimized performs randomized SVD
# In both cases, requested number of largest singular values is returned

# Generate test signal (real signal with three harmonics)

length = 1024
time = torch.linspace(1, length, length, dtype=dtype, device=device)
signal = 1.0*torch.sin(2*numpy.pi*0.12*time) + 0.1*torch.sin(2*numpy.pi*0.24*time) + 1.E-3*torch.sin(2*numpy.pi*0.31*time)
signal.unsqueeze_(0)

# Generate matrix representation

matrix = Filter.make_matrix(signal)

# Since, test signal contains three harmonics, it's matrix is expected to have six non-zero singular values

print(torch.linalg.matrix_rank(matrix))

# Compute singular values (full SVD)

S1 = Filter.svd_list(rank=6, matrix=matrix, cpu=True)

# svd_list used full SVD to compute singular values
# number of returned singular values is set by rank parameter
# input matrix should be a batch of matrices
# cpu flag is used to compute SVD using CPU

# Compute singular values (randomized SVD)

S2 = Filter.svd_list_randomized(rank=6, matrix=matrix, buffer=8, count=8, cpu=True)

# svd_list_randomized used randomized SVD to compute singular values
# randomized_range function is used to estimate matrix range with QR decomposition
# buffer parameter sets the number of extra dimensions, randomized range column rank is rank + buffer
# count parameter sets number of iterations

# Compare singular values

fmt = 3 * '{:<20.12}'
for s1, s2 in zip(S1.squeeze().cpu(), S2.squeeze().cpu()):
    print(fmt.format(s1, s2, abs(s1 - s2)))

tensor([6])
256.518221536       256.518221536       5.68434188608e-14   
255.980457668       255.980457668       2.84217094304e-14   
25.6585560702       25.6585560702       7.1054273576e-15    
25.5899723888       25.5899723888       1.06581410364e-14   
0.256513879015      0.256513879015      3.88578058619e-16   
0.255971936601      0.255971936601      1.66533453694e-16   
